In [ ]:
%pip install tensorflow

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

In [ ]:
#keymixer

In [ ]:
class KeyMixer(layers.Layer):

    def __init__(self):
        super().__init__()

        self.relu = layers.ReLU()
        self.sigmoid = layers.Activation("sigmoid")

    def build(self, input_shape):

        channels = input_shape[-1]

        # Trainable matrices
        self.W1 = self.add_weight(
            shape=(channels, channels),
            initializer="glorot_uniform",
            trainable=True,
            name="W1"
        )

        self.b1 = self.add_weight(
            shape=(channels,),
            initializer="zeros",
            trainable=True,
            name="b1"
        )

        self.W2 = self.add_weight(
            shape=(channels, channels),
            initializer="glorot_uniform",
            trainable=True,
            name="W2"
        )

        self.b2 = self.add_weight(
            shape=(channels,),
            initializer="zeros",
            trainable=True,
            name="b2"
        )

        # Fixed random tensor K
        self.K = self.add_weight(
            shape=input_shape[1:],
            initializer=tf.keras.initializers.RandomNormal(
                mean=0.0,
                stddev=0.01
            ),
            trainable=False,
            name="K"
        )

    def call(self, S):

        x = tf.matmul(S, self.W1) + self.b1
        x = self.relu(x)

        x = tf.matmul(x, self.W2) + self.b2

        x = self.sigmoid(x)

        x = x + self.K

        x = tf.clip_by_value(x, 0.0, 1.0)

        return x

In [ ]:
key_mixer = KeyMixer()

secret = tf.random.uniform(
    (1, 96, 96, 3),
    minval=0,
    maxval=1
)

transformed_secret = key_mixer(secret)

print("Secret shape:     ", secret.shape)
print("Transformed shape:", transformed_secret.shape)
print("Minimum value:    ", tf.reduce_min(transformed_secret).numpy())
print("Maximum value:    ", tf.reduce_max(transformed_secret).numpy())

In [ ]:
#residual block

In [ ]:
def residual_block(x, filters):
    shortcut = x

    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(
        filters, (3, 3),
        padding="same"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(
        filters, (3, 3),
        padding="same"
    )(x)

    # Match channels if required
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(
            filters, (1, 1),
            padding="same"
        )(shortcut)

    x = layers.Add()([x, shortcut])

    return x

In [ ]:
#encoder

In [ ]:
def build_encoder():

    inputs = layers.Input(shape=(96, 96, 6))

    # Conv1
    x = layers.Conv2D(
        64, (7, 7),
        padding="same"
    )(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # Save 96×96 features
    skip1 = x

    # Down1 → 48×48
    x = layers.Conv2D(
        128, (3, 3),
        strides=2,
        padding="same"
    )(x)

    x = residual_block(x, 128)
    x = residual_block(x, 128)

    # Save 48×48 features
    skip2 = x

    # Down2 → 24×24
    x = layers.Conv2D(
        256, (3, 3),
        strides=2,
        padding="same"
    )(x)

    x = residual_block(x, 256)
    x = residual_block(x, 256)

    # Up1 → 48×48
    x = layers.Conv2DTranspose(
        128, (3, 3),
        strides=2,
        padding="same"
    )(x)

    # Combine with saved 48×48 features
    x = layers.Concatenate()([x, skip2])

    # Reduce back to 128 channels
    x = layers.Conv2D(
        128, (1, 1),
        padding="same"
    )(x)

    x = residual_block(x, 128)
    x = residual_block(x, 128)

    # Up2 → 96×96
    x = layers.Conv2DTranspose(
        64, (3, 3),
        strides=2,
        padding="same"
    )(x)

    # Combine with saved 96×96 features
    x = layers.Concatenate()([x, skip1])

    x = layers.Conv2D(
        64, (1, 1),
        padding="same"
    )(x)

    x = residual_block(x, 64)
    x = residual_block(x, 64)

    # Output
    outputs = layers.Conv2D(
        3, (7, 7),
        padding="same",
        activation="sigmoid"
    )(x)

    return Model(inputs, outputs, name="Paper_Encoder")

In [ ]:
encoder = build_encoder()

encoder.summary()

In [ ]:
dummy_input = tf.random.normal((1, 96, 96, 6))

dummy_output = encoder(dummy_input)

print("Input :", dummy_input.shape)
print("Output:", dummy_output.shape)

In [ ]:
#decoder

In [ ]:
def build_decoder():

    # Input: Container image
    inputs = layers.Input(shape=(96, 96, 3))

    # Conv1
    x = layers.Conv2D(
        64, (7, 7),
        strides=1,
        padding="same"
    )(inputs)

    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # Down1
    x = layers.Conv2D(
        128, (3, 3),
        strides=2,
        padding="same"
    )(x)

    x = residual_block(x, 128)
    x = residual_block(x, 128)

    # Down2
    x = layers.Conv2D(
        256, (3, 3),
        strides=2,
        padding="same"
    )(x)

    x = residual_block(x, 256)
    x = residual_block(x, 256)

    # Up1
    x = layers.Conv2DTranspose(
        128, (3, 3),
        strides=2,
        padding="same"
    )(x)

    x = residual_block(x, 128)
    x = residual_block(x, 128)

    # Up2
    x = layers.Conv2DTranspose(
        64, (3, 3),
        strides=2,
        padding="same"
    )(x)

    x = residual_block(x, 64)
    x = residual_block(x, 64)

    # Output
    outputs = layers.Conv2D(
        3, (7, 7),
        strides=1,
        padding="same",
        activation="sigmoid"
    )(x)

    return Model(
        inputs,
        outputs,
        name="Paper_Decoder"
    )

In [ ]:
decoder = build_decoder()

decoder.summary()

In [ ]:
dummy_container = tf.random.normal((1, 96, 96, 3))

dummy_recovered = decoder(dummy_container)

print("Input shape:", dummy_container.shape)
print("Output shape:", dummy_recovered.shape)

In [ ]:
#connect


In [ ]:
class PaperCNNModel(Model):

    def __init__(self):
        super().__init__()

        self.keymixer = KeyMixer()
        self.encoder = build_encoder()
        self.decoder = build_decoder()

    def call(self, inputs):

        cover, secret = inputs

        # 1. Transform secret
        transformed_secret = self.keymixer(secret)

        # 2. Combine cover + transformed secret
        combined = tf.concat(
            [cover, transformed_secret],
            axis=-1
        )

        # 3. Generate container/stego image
        container = self.encoder(combined)

        # 4. Recover secret
        recovered_secret = self.decoder(container)

        return container, recovered_secret

In [ ]:
model = PaperCNNModel()

In [ ]:
cover = tf.random.uniform((1, 96, 96, 3))
secret = tf.random.uniform((1, 96, 96, 3))

container, recovered_secret = model((cover, secret))

print("Cover:", cover.shape)
print("Secret:", secret.shape)
print("Container:", container.shape)
print("Recovered secret:", recovered_secret.shape)

In [ ]:
#stl 10


In [ ]:
!pip install tensorflow-datasets

In [ ]:
import tensorflow_datasets as tfds

(train_ds, test_ds), info = tfds.load(
    "stl10",
    split=["train", "test"],
    as_supervised=True,
    with_info=True
)

print(info)

In [ ]:
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image


train_images = train_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)

test_images = test_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)

In [ ]:
BATCH_SIZE = 32

cover_ds = (
    train_images
    .shuffle(5000)
    .repeat()
)

secret_ds = (
    train_images
    .shuffle(5000)
    .repeat()
)

paired_train = tf.data.Dataset.zip(
    (cover_ds, secret_ds)
)

paired_train = paired_train.batch(BATCH_SIZE)
paired_train = paired_train.prefetch(tf.data.AUTOTUNE)

In [ ]:
def image_loss(y_true, y_pred):

    # Mean Squared Error
    mse = tf.reduce_mean(
        tf.square(y_true - y_pred)
    )

    # SSIM
    ssim = tf.reduce_mean(
        tf.image.ssim(
            y_true,
            y_pred,
            max_val=1.0
        )
    )

    # Convert SSIM into a minimizable loss
    ssim_loss = 1.0 - ssim

    return mse + ssim_loss

In [ ]:
container, recovered_secret

In [ ]:
model = PaperCNNModel()

In [ ]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.0001,
    beta_1=0.9,
    beta_2=0.999
)

model.compile(
    optimizer=optimizer,
    loss=[
        image_loss,
        image_loss
    ],
    loss_weights=[
        1.0,
        5.0
    ]
)

In [ ]:
def prepare_training_pair(cover, secret):
    return (cover, secret), (cover, secret)


train_data = paired_train.map(
    prepare_training_pair,
    num_parallel_calls=tf.data.AUTOTUNE
)

In [ ]:
Input:
    cover
    secret

Target:
    cover
    secret

In [ ]:
history = model.fit(
    train_data,
    steps_per_epoch=5,
    epochs=1
)

In [ ]:
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="loss",
    factor=0.5,
    patience=5,
    verbose=1
)


In [ ]:
# Split STL-10 training images
train_base = train_images.take(4500)
val_base = train_images.skip(4500)

In [ ]:
def make_pairs(dataset):
    cover = dataset.shuffle(4500).repeat()
    secret = dataset.shuffle(4500).repeat()

    pairs = tf.data.Dataset.zip((cover, secret))

    pairs = pairs.map(
        prepare_training_pair,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    pairs = pairs.batch(BATCH_SIZE)
    pairs = pairs.prefetch(tf.data.AUTOTUNE)

    return pairs

In [ ]:
train_data = make_pairs(train_base)
val_data = make_pairs(val_base)

In [ ]:
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    verbose=1
)

In [ ]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "paper1_best_model.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

In [ ]:
steps_per_epoch = 4500 // 32
validation_steps = 500 // 32

print("Training steps:", steps_per_epoch)
print("Validation steps:", validation_steps)

In [ ]:
history = model.fit(
    train_data,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_data,
    validation_steps=validation_steps,
    epochs=50,
    callbacks=[
        lr_scheduler,
        checkpoint
    ]
)

In [ ]:
test_pairs = tf.data.Dataset.zip(
    (
        test_images,
        test_images.skip(1)
    )
)

test_pairs = test_pairs.map(
    prepare_training_pair,
    num_parallel_calls=tf.data.AUTOTUNE
)

test_pairs = test_pairs.batch(32)
test_pairs = test_pairs.prefetch(tf.data.AUTOTUNE)

In [ ]:
def calculate_metrics(model, dataset):

    cover_psnr_values = []
    secret_psnr_values = []

    cover_ssim_values = []
    secret_ssim_values = []

    for (cover, secret), (cover_target, secret_target) in dataset:

        container, recovered_secret = model(
            (cover, secret),
            training=False
        )

        # PSNR
        cover_psnr = tf.image.psnr(
            cover_target,
            container,
            max_val=1.0
        )

        secret_psnr = tf.image.psnr(
            secret_target,
            recovered_secret,
            max_val=1.0
        )

        # SSIM
        cover_ssim = tf.image.ssim(
            cover_target,
            container,
            max_val=1.0
        )

        secret_ssim = tf.image.ssim(
            secret_target,
            recovered_secret,
            max_val=1.0
        )

        cover_psnr_values.extend(cover_psnr.numpy())
        secret_psnr_values.extend(secret_psnr.numpy())

        cover_ssim_values.extend(cover_ssim.numpy())
        secret_ssim_values.extend(secret_ssim.numpy())

    return {
        "Cover PSNR": sum(cover_psnr_values) / len(cover_psnr_values),
        "Secret PSNR": sum(secret_psnr_values) / len(secret_psnr_values),
        "Cover SSIM": sum(cover_ssim_values) / len(cover_ssim_values),
        "Secret SSIM": sum(secret_ssim_values) / len(secret_ssim_values)
    }

In [ ]:
results = calculate_metrics(model, test_pairs)

results

In [ ]:
import matplotlib.pyplot as plt

for (cover, secret), _ in test_pairs.take(1):

    container, recovered_secret = model(
        (cover, secret),
        training=False
    )

    plt.figure(figsize=(12, 8))

    plt.subplot(2, 2, 1)
    plt.imshow(cover[0])
    plt.title("Original Cover")
    plt.axis("off")

    plt.subplot(2, 2, 2)
    plt.imshow(container[0])
    plt.title("Container / Stego")
    plt.axis("off")

    plt.subplot(2, 2, 3)
    plt.imshow(secret[0])
    plt.title("Original Secret")
    plt.axis("off")

    plt.subplot(2, 2, 4)
    plt.imshow(recovered_secret[0])
    plt.title("Recovered Secret")
    plt.axis("off")

    plt.show()